## Извлекаем датасет

In [1]:
!ls -lh .

итого 956M
-rw-rw-r-- 1 lev lev 320M апр 22 01:08 best_cat_breed_model.pth
-rw-rw-r-- 1 lev lev  15K апр 20 11:43 CatBreedAI.ipynb
-rw-rw-r-- 1 lev lev 637M апр 19 07:31 data.tar.gz
-rw-rw-r-- 1 lev lev    0 апр 22 00:59 README.md


In [6]:
!tar -xzf data.tar.gz -C .

In [11]:
!ls data

train  val


## Подготовка данных

In [14]:
%pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 892.8 kB/s eta 0:00:00 eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.4 MB/s eta 0:00:00m eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 2.4 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 1.9 MB/s eta 0:00:00m eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 2.8 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 2.1 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.1 MB/s eta 0:00:00m eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 3.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 3.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 5.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━

In [15]:
%pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 329.9 kB/s eta 0:00:0031m2.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 888.6 kB/s eta 0:00:00eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 315.7 kB/s eta 0:00:00 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 297.8 kB/s eta 0:00:00 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 2.2 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 2.2 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 3.3 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 904.4 kB/s eta 0:00:001m6.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 5.1 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 656.9 kB/s eta 0:00:001m22.9 MB/s eta 0:00:01
   ━━━━━━

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm
import os
from PIL import ImageFile, Image
from pathlib import Path
import shutil
import numpy as np

import warnings
warnings.filterwarnings('ignore')

/home/lev/Projects/catbreed-helper/ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = './data'
BEST_MODEL_PATH = 'models/best_cat_breed_model.pth'
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

cpu


In [3]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
# warnings.filterwarnings('ignore', category=UserWarning, module='PIL')

class SafeImageFolder(datasets.ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        try:
            sample = self.loader(path)
            if self.transform is not None:
                sample = self.transform(sample)
            return sample, target
        except Exception as e:
            print(f"Error loading {path}: {e}, skipping...")
            # Возвращаем случайный другой элемент
            new_index = (index + 1) % len(self.samples)
            return self.__getitem__(new_index)

## Проверка файлов

```
import os
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms

# Трансформации
transform = transforms.Compose([
    transforms.RandomResizedCrop(384),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

data_dir = '/content/data/train'

for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            path = os.path.join(root, file)
            try:
                with Image.open(path) as img:
                    img = img.convert('RGB')
                    # Применяем трансформации
                    tensor = transform(img)
            except Exception as e:
                print(f"❌ БИТЫЙ: {path}")
                print(f"   Ошибка: {e}")
```

In [3]:
# Битое изображение
# Проверку закомментировал т.к. занимает много времени
try:
  os.remove('/content/data/train/somali/33477513_41.jpg')
except:
  pass

In [4]:
# Трансформации
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),  # Разнообразие масштаба
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),  # Иногда отражение по вертикали
    transforms.RandomRotation(degrees=20),  # Повороты ±20 градусов
    transforms.ColorJitter(
        brightness=0.3,   # Разная яркость
        contrast=0.3,     # Разный контраст
        saturation=0.3,   # Разная насыщенность
        hue=0.1           # Разный оттенок
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),  # Сдвиги
        scale=(0.9, 1.1),      # Масштаб
        shear=10                # Искажение
    ),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),  # Перспектива
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # Случайные прямоугольники
])

val_transform = transforms.Compose([
  transforms.Resize(int(IMG_SIZE * 1.14)),
  transforms.CenterCrop(IMG_SIZE),
  transforms.ToTensor(),
  transforms.Normalize(
      mean=[0.485, 0.456, 0.406],
      std=[0.229, 0.224, 0.225]
  )
])

# Загружаем список файлов
# train_dataset = datasets.ImageFolder(
#   root=os.path.join(DATA_DIR, 'train'),
#   transform=train_transform
# )

# val_dataset = datasets.ImageFolder(
#   root=os.path.join(DATA_DIR, 'val'),
#   transform=val_transform
# )
train_dataset = SafeImageFolder(
  root=os.path.join(DATA_DIR, 'train'),
  transform=train_transform
)

val_dataset = SafeImageFolder(
  root=os.path.join(DATA_DIR, 'val'),
  transform=val_transform
)
    
# DataLoader подгружает только текущий батч
train_loader = DataLoader(
  train_dataset, batch_size=BATCH_SIZE, shuffle=True,
  num_workers=0, pin_memory=True
)

val_loader = DataLoader(
  val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
  num_workers=0, pin_memory=True
)

In [6]:
NUM_CLASSES = len(train_dataset.classes)
print(f"Классов: {NUM_CLASSES}")
print(f"Классы: {train_dataset.classes[:5]}...")

with open('classes.txt', 'w') as f:
    f.writelines([ s + '\n' for s in train_dataset.classes ])

Классов: 66
Классы: ['abyssinian', 'american_bobtail', 'american_curl', 'american_shorthair', 'american_wirehair']...


## Модель
Используем предобученную `ConvNeXt-Tiny`

In [21]:
DEVICE

device(type='cpu')

In [22]:
model = timm.create_model(
    'convnext_tiny',  # Веса ImageNet-1K
    pretrained=True,           
    num_classes=NUM_CLASSES,
    drop_rate=0.0,
    drop_path_rate=0.1,
)

model = model.to(DEVICE)

if os.path.exists(BEST_MODEL_PATH):
    state = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state['model_state_dict'])
    print('model loaded from file')

# Регуляризация
# original_head = model.head
# model.head = nn.Sequential(
#     nn.Dropout(0.5),
#     nn.Linear(original_head.in_features, NUM_CLASSES)
# )

In [23]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Разный lr для частей
optimizer = optim.AdamW(
  [
    {'params': model.stem.parameters(), 'lr': LEARNING_RATE * 0.01},
    {'params': model.stages.parameters(), 'lr': LEARNING_RATE * 0.05},
    {'params': model.head.parameters(), 'lr': LEARNING_RATE}  # Head учится быстрее
  ],
  weight_decay=0.1
)

# Динамический lr
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Ускорение
if DEVICE == 'cuda':
  scaler = torch.amp.GradScaler('cuda')
else:
  scaler = None

## Подготовка к обучению

In [25]:
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1-lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

In [28]:
def train_one_epoch():
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.2)
        
        optimizer.zero_grad()
        
        if DEVICE == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = lam * criterion(outputs, labels_a) + (1-lam) * criterion(outputs, labels_b)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        else:
            # Обычный проход для CPU
            outputs = model(images)
            # loss = criterion(outputs, labels)
            loss = lam * criterion(outputs, labels_a) + (1-lam) * criterion(outputs, labels_b)
            loss.backward()
            optimizer.step()
    
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
          'loss': f'{running_loss/(pbar.n+1):.3f}',
          'acc': f'{100.*correct/total:.1f}%'
        })
    
    return running_loss / len(train_loader), 100. * correct / total

In [29]:
def validate():
  model.eval()
  running_loss = 0.0
  correct = 0
  total = 0

  with torch.no_grad():
    pbar = tqdm(val_loader, desc='Validation')
    for images, labels in pbar:
      images, labels = images.to(DEVICE), labels.to(DEVICE)

      with torch.cuda.amp.autocast():
        outputs = model(images)
        loss = criterion(outputs, labels)

      running_loss += loss.item()
      _, predicted = outputs.max(1)
      total += labels.size(0)
      correct += predicted.eq(labels).sum().item()

      pbar.set_postfix({
        'loss': f'{running_loss/(pbar.n+1):.3f}',
        'acc': f'{100.*correct/total:.1f}%'
      })

    return running_loss / len(val_loader), 100. * correct / total

## Обучение

In [30]:
!mkdir -p models

In [ ]:
best_acc = 0.0

epoch = 0
while epoch < NUM_EPOCHS:
    filename = f'models/cat_breed_model_{epoch+1}.pth'
    while os.path.exists(filename):
        print(f'{filename} exist, next epoch')
        epoch += 1
        filename = f'models/cat_breed_model_{epoch+1}.pth'

    print(f"\nEpoch: {epoch+1}/{NUM_EPOCHS}")

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()
    
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    torch.save(
      {
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc':              val_acc,
        'classes':              train_dataset.classes
      },
      filename
    )
    # Сохраняем лучшую модель
    if val_acc > best_acc:
        best_acc = val_acc
    
        shutil.copy2(
            filename,
            BEST_MODEL_PATH
        )
    epoch += 1


Epoch: 1/15


Training:   5%|▎     | 54/1192 [08:40<2:56:56,  9.33s/it, loss=3.778, acc=11.1%]

In [13]:
best_acc

61.50224918924574